# Message Intelligence System
## Objective:
**This project is to designa classification System that identifies whether incoming messages are spam or Legitimate**

In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

In [2]:
base_dir = Path().cwd().parent

dataset_path = base_dir/"data"/"Message_Intelligence_Dataset.csv"
model_dir = base_dir/"model"
output_dir  = base_dir/"outputs"

# Dataset Understanding & Preparation

In [3]:
df= pd.read_csv(dataset_path)

display(df.head())

print("\nInfo:")
print(df.info())

print("\nshape:")
print(df.shape)

,message_id,message_text,message_length,word_count,num_urls,num_digits,num_special_chars,spam_keyword_score,legit_keyword_score,sender_activity_score,sender_account_age_days,messages_sent_last_24h,timestamp,hour_of_day,day_of_week,spam_label
0,900001,Please find the attached invoice for the updat...,99,11,1,4,0,0,1,56.6,500.0,6.0,2025-11-24 02:00:00,2,0,0
1,900002,Let's catch up tomorrow regarding the timeline...,73,12,0,0,0,0,0,16.6,207.0,0.0,2025-12-17 21:00:00,21,2,0
2,900003,Can you send the report by end of day? next Mo...,67,13,0,0,0,0,1,25.7,418.0,6.0,2025-11-15 13:00:00,13,5,0
3,900004,Can you send the report by end of day? 10:30 A...,64,13,0,4,0,0,1,48.8,276.0,5.0,2025-12-17 23:00:00,23,2,0
4,900005,Could you review the document and share feedba...,84,14,0,0,0,0,1,33.0,683.0,7.0,2025-11-29 11:00:00,11,5,0



Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5200 entries, 0 to 5199
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   message_id               5200 non-null   int64  
 1   message_text             5200 non-null   object 
 2   message_length           5200 non-null   int64  
 3   word_count               5200 non-null   int64  
 4   num_urls                 5200 non-null   int64  
 5   num_digits               5200 non-null   int64  
 6   num_special_chars        5200 non-null   int64  
 7   spam_keyword_score       5200 non-null   int64  
 8   legit_keyword_score      5200 non-null   int64  
 9   sender_activity_score    5094 non-null   float64
 10  sender_account_age_days  5087 non-null   float64
 11  messages_sent_last_24h   5038 non-null   float64
 12  timestamp                5200 non-null   object 
 13  hour_of_day              5200 non-null   int64  
 14  day_of_week      

In [4]:
# Missing and Dupicated values
print("Missing values count:\n",df.isnull().sum())

print("\nDuplicate count:",df.duplicated().sum())

Missing values count:
 message_id                   0
message_text                 0
message_length               0
word_count                   0
num_urls                     0
num_digits                   0
num_special_chars            0
spam_keyword_score           0
legit_keyword_score          0
sender_activity_score      106
sender_account_age_days    113
messages_sent_last_24h     162
timestamp                    0
hour_of_day                  0
day_of_week                  0
spam_label                   0
dtype: int64

Duplicate count: 0


In [5]:
# Target variable analysis
target_counts = df["spam_label"].value_counts().reindex([0,1],fill_value=0)
target_PT = (df["spam_label"].value_counts(normalize=True).reindex([0,1],fill_value=0)*100).round(2)

target_summary = pd.DataFrame({
    "Message_Type":["Legitimete","Spam"],
    "Count":target_counts.values,
    "Percentage":[f"{x}%" for x in target_PT.values]
})
display(target_summary)

,Message_Type,Count,Percentage
0,Legitimete,4227,81.29%
1,Spam,973,18.71%


In [6]:
df.columns

Index(['message_id', 'message_text', 'message_length', 'word_count',
       'num_urls', 'num_digits', 'num_special_chars', 'spam_keyword_score',
       'legit_keyword_score', 'sender_activity_score',
       'sender_account_age_days', 'messages_sent_last_24h', 'timestamp',
       'hour_of_day', 'day_of_week', 'spam_label'],
      dtype='object')

In [7]:
# select X and y
X_cols = df.drop(columns=["message_id","message_text","timestamp","spam_label"]).columns
target_col = "spam_label"
required_cols= X_cols.tolist() + ["spam_label"]
required_cols

['message_length',
 'word_count',
 'num_urls',
 'num_digits',
 'num_special_chars',
 'spam_keyword_score',
 'legit_keyword_score',
 'sender_activity_score',
 'sender_account_age_days',
 'messages_sent_last_24h',
 'hour_of_day',
 'day_of_week',
 'spam_label']

In [8]:
X = df[X_cols].copy()
y=df[target_col].copy()

print("X shape:",X.shape)
print("y shape:",y.shape)

X shape: (5200, 12)
y shape: (5200,)


In [9]:
# Train test Split
X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42,stratify=y)

print("X train Shape:",X_train.shape)
print("X test Shape:",X_test.shape)
print("y train Shape:",y_train.shape)
print("y test Shape:",y_test.shape)

X train Shape: (4160, 12)
X test Shape: (1040, 12)
y train Shape: (4160,)
y test Shape: (1040,)


In [12]:
X_cols

Index(['message_length', 'word_count', 'num_urls', 'num_digits',
       'num_special_chars', 'spam_keyword_score', 'legit_keyword_score',
       'sender_activity_score', 'sender_account_age_days',
       'messages_sent_last_24h', 'hour_of_day', 'day_of_week'],
      dtype='object')

In [13]:
# create Preprocessing pipelines

# this for KNN and SVM
scaled_preprocessor = Pipeline([
    ("imputer",SimpleImputer(strategy="median")),
    ("scaler",StandardScaler())
])

per_KNN_SVM=ColumnTransformer([
    ("columns",scaled_preprocessor,X_cols)
])
# this for Gaussian Naive Bayes
nb_preprocessor = Pipeline([
    ("imputer",SimpleImputer(strategy="median"))
])

per_NB=ColumnTransformer([
    ("columns",nb_preprocessor,X_cols)
])


In [48]:
def evaluate_model(model,X_test,y_test,model_name,pos_label=1):
    # prediction
    y_pred = model.predict(X_test)

    # Confusion Matrix
    tn,fp,fn,tp = confusion_matrix(
    y_test,y_pred,labels=[0,1]).ravel()

    # Performance metrics
    results={
    "Model":model_name,
    "Accuracy":accuracy_score(y_test,y_pred),
    "Precision":precision_score(y_test,y_pred,pos_label=pos_label,zero_division=0),
    "recall":recall_score(y_test,y_pred,pos_label=pos_label,zero_division=0),
    "F1":f1_score(y_test,y_pred,pos_label=pos_label,zero_division=0),
    "TN":tn,
    "FP":fp,
    "FN":fn,
    "TP":tp
    }

    return results

# Baseline Model – K-Nearest Neighbors

In [49]:
Base_KNN = Pipeline([
    ("preprocesser",per_KNN_SVM),
    ("model",KNeighborsClassifier())
])

In [50]:
param_grid_knn = {
    "model__n_neighbors":[3,5,7,9],
    "model__weights":["uniform","distance"],
    "model__metric":["euclidean","manhattan"]
}

In [51]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# KNN GridSearchCV
grid_KNN = GridSearchCV(
    estimator=Base_KNN,
    param_grid=param_grid_knn,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    refit=True
)

grid_KNN.fit(X_train,y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...lassifier())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__metric': ['euclidean', 'manhattan'], 'model__n_neighbors': [3, 5, ...], 'model__weights': ['uniform', 'distance']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for c

In [52]:

cv_results_KNN = pd.DataFrame(grid_KNN.cv_results_)

distance_analysis = (
    cv_results_KNN[
        [
            "param_model__n_neighbors",
            "param_model__weights",
            "param_model__metric",
            "mean_test_score"
        ]
    ]
    .sort_values("mean_test_score", ascending=False)
)

display(distance_analysis)


,param_model__n_neighbors,param_model__weights,param_model__metric,mean_test_score
8,3,uniform,manhattan,1.000000
9,3,distance,manhattan,1.000000
10,5,uniform,manhattan,1.000000
11,5,distance,manhattan,1.000000
13,7,distance,manhattan,1.000000
0,3,uniform,euclidean,0.999353
1,3,distance,euclidean,0.999353
2,5,uniform,euclidean,0.999353
3,5,distance,euclidean,0.999353
4,7,uniform,euclidean,0.999353


In [53]:
# Step 1: Predict
y_pred_KNN = best_KNN.predict(X_test)

# Step 2: Create a copy of test data
misclassified = X_test.copy()

# Step 3: Add actual and predicted labels
misclassified["Actual"] = y_test
misclassified["Predicted"] = y_pred_KNN

# Step 4: Keep only wrong predictions
misclassified = misclassified[
    misclassified["Actual"] != misclassified["Predicted"]
]

# Step 5: Display first 10 wrong predictions
display(misclassified.head(10))

,message_length,word_count,num_urls,num_digits,num_special_chars,spam_keyword_score,legit_keyword_score,sender_activity_score,sender_account_age_days,messages_sent_last_24h,hour_of_day,day_of_week,Actual,Predicted


In [54]:
result_KNN = pd.DataFrame({
        "Model":"KNN",
        "Best Parameters":grid_KNN.best_params_,
        "Best CV F1":grid_KNN.best_score_
})
display(result_KNN)
best_KNN = grid_KNN.best_estimator_

,Model,Best Parameters,Best CV F1
model__metric,KNN,manhattan,1.0
model__n_neighbors,KNN,3,1.0
model__weights,KNN,uniform,1.0


In [55]:
# test results
test_result = evaluate_model(best_KNN,X_test,y_test,"KNN")

test_df_KNN = pd.DataFrame(test_result,index=[0])

display(test_df_KNN)

,Model,Accuracy,Precision,recall,F1,TN,FP,FN,TP
0,KNN,1.0,1.0,1.0,1.0,845,0,0,195


# Support Vector Machine Classifier

In [56]:
Base_svm = Pipeline([
    ("preprocesser",per_KNN_SVM),
    ("model",SVC())
])

In [57]:
param_grid_SVC = {
    "model__C":[0.1,1,10,100],
    "model__kernel":["linear","rbf"],
    "model__gamma":["scale","auto"]
}

In [68]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# SVC GridSearchCV
grid_SVC = GridSearchCV(
    estimator=Base_svm,
    param_grid=param_grid_SVC,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    refit=True
)

grid_SVC.fit(X_train,y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...del', SVC())])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__C': [0.1, 1, ...], 'model__gamma': ['scale', 'auto'], 'model__kernel': ['linear', 'rbf']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"

In [59]:
result_SVC = pd.DataFrame({
        "Model":"SVC",
        "Best Parameters":grid_SVC.best_params_,
        "Best CV F1":grid_SVC.best_score_
})
display(result_SVC)
best_SVC = grid_SVC.best_estimator_

,Model,Best Parameters,Best CV F1
model__C,SVC,0.1,1.0
model__gamma,SVC,scale,1.0
model__kernel,SVC,linear,1.0


In [64]:
# Access the fitted SVC model inside the pipeline
svc_model = best_SVC.named_steps["model"]

# Analyze support vectors
print("Best SVM Parameters:", grid_SVC.best_params_)
print("Number of Support Vectors:", svc_model.n_support_)
print("Total Support Vectors:", svc_model.n_support_.sum())
print("Number of Support Vectors per Class:", svc_model.n_support_.tolist())


Best SVM Parameters: {'model__C': 0.1, 'model__gamma': 'scale', 'model__kernel': 'linear'}
Number of Support Vectors: [9 9]
Total Support Vectors: 18
Number of Support Vectors per Class: [9, 9]


In [60]:
# test results
test_result = evaluate_model(best_SVC,X_test,y_test,"SVC")

test_df_SVC = pd.DataFrame(test_result,index=[0])

display(test_df_SVC)

,Model,Accuracy,Precision,recall,F1,TN,FP,FN,TP
0,SVC,1.0,1.0,1.0,1.0,845,0,0,195


In [63]:
# KNN vs SVC
comparison = pd.concat([test_df_KNN,test_df_SVC],ignore_index=True)
display(comparison.sort_values(by="F1",ascending=False))

,Model,Accuracy,Precision,recall,F1,TN,FP,FN,TP
0,KNN,1.0,1.0,1.0,1.0,845,0,0,195
1,SVC,1.0,1.0,1.0,1.0,845,0,0,195


# Naive Bayes Classifier & Probability

In [65]:
Base_NB = Pipeline([
    ("preprocesser",per_NB),
    ("model",GaussianNB())
])

In [67]:
param_grid_NB = {
    "model__var_smoothing":[
        1e-11,1e-10,1e-9,1e-8,1e-7
    ]
}

In [69]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# NB GridSearchCV
grid_NB = GridSearchCV(
    estimator=Base_NB,
    param_grid=param_grid_NB,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    refit=True
)

grid_NB.fit(X_train,y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...aussianNB())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__var_smoothing': [1e-11, 1e-10, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, default=0Controls the verbosity o

In [70]:
result_NB = pd.DataFrame({
        "Model":"GaussianNB",
        "Best Parameters":grid_NB.best_params_,
        "Best CV F1":grid_NB.best_score_
})
display(result_NB)
best_NB = grid_NB.best_estimator_

,Model,Best Parameters,Best CV F1
model__var_smoothing,GaussianNB,1.000000e-11,1.0


In [71]:
# test results
test_result = evaluate_model(best_NB,X_test,y_test,"GaussianNB")

test_df_NB = pd.DataFrame(test_result,index=[0])

display(test_df_NB)

,Model,Accuracy,Precision,recall,F1,TN,FP,FN,TP
0,GaussianNB,1.0,1.0,1.0,1.0,845,0,0,195


# Model Comparison & Evaluation

In [88]:
all_model_metric = pd.concat([test_df_KNN,test_df_SVC,test_df_NB],ignore_index=True)

display(all_model_metric)

,Model,Accuracy,Precision,recall,F1,TN,FP,FN,TP
0,KNN,1.0,1.0,1.0,1.0,845,0,0,195
1,SVC,1.0,1.0,1.0,1.0,845,0,0,195
2,GaussianNB,1.0,1.0,1.0,1.0,845,0,0,195


# Final Analysis & Reporting

## 1. Strengths and Weaknesses of Each Classifier

### K-Nearest Neighbors (KNN)

**Strengths:**
- KNN is simple and easy to understand.
- It classifies a message based on the nearest training samples.
- It can capture complex decision boundaries.

**Weaknesses:**
- Prediction can be slow on large datasets because it calculates distances to training samples.
- It is sensitive to feature scaling and the choice of distance metric.
- Its performance can decrease when there are many irrelevant features.

### Support Vector Machine (SVM)

**Strengths:**
- SVM can create effective decision boundaries using Linear and RBF kernels.
- It can perform well with high-dimensional datasets.
- It uses support vectors to define the decision boundary.

**Weaknesses:**
- Selecting the correct kernel and hyperparameters can be difficult.
- Training can be expensive on large datasets.
- Its decision process is less interpretable than a simple model.
- Overlapping classes can make perfect separation impossible, but SVM can still find a useful boundary.

### Naive Bayes (NB)

**Strengths:**
- Naive Bayes is fast to train and predict.
- It provides class probabilities.
- Multinomial Naive Bayes is a strong baseline for text classification using word counts or TF-IDF features.
- It works well in many high-dimensional text classification problems.

**Weaknesses:**
- It assumes that features are conditionally independent given the class.
- Correlated features can make its probability estimates less reliable.
- Its performance depends on whether the selected Naive Bayes variant matches the data.

## 2. Impact of Probability Assumptions in Naive Bayes

Naive Bayes uses Bayes' Theorem to calculate the probability of a class given the observed features.

Its main assumption is that features are conditionally independent given the class.

For example, in spam classification, the words "free" and "prize" may frequently occur together. Naive Bayes treats their contributions as conditionally independent, which may not fully represent their real relationship.

GaussianNB assumes continuous features follow Gaussian distributions within each class. MultinomialNB is generally more suitable for non-negative word-count or TF-IDF features.

These assumptions make Naive Bayes efficient, but its predicted probabilities may not always be well calibrated.

## 3. Trade-offs Between Interpretability and Performance

- **KNN:** Easy to explain using nearest examples, but predictions can be slow and depend on feature scaling.
- **SVM:** Can achieve strong predictive performance, but understanding its decision boundary is more difficult, especially with an RBF kernel.
- **Naive Bayes:** Relatively easy to explain using class priors and feature likelihoods, and it is computationally efficient. However, its independence assumptions may limit performance on some datasets.

The final choice should balance interpretability, prediction quality, speed, and maintenance requirements.

## 4. Business Recommendation for Real-World Deployment

For a real-world spam detection system, I would evaluate all three classifiers using the same held-out test dataset.

I would compare:
- Precision
- Recall
- F1-score
- Confusion matrix
- Prediction latency

False positives are important because a legitimate message classified as spam may cause a user to miss an important notification. False negatives are also important because spam messages may reach users.

I would initially consider Multinomial Naive Bayes as a fast baseline for direct text classification, and compare it with a linear SVM. KNN would remain a benchmark if its prediction speed and performance were acceptable.

The final model should be selected based on measured test results and business priorities, rather than assuming that one algorithm is always the best.

Before deployment, I would also monitor model performance, review misclassified messages, and retrain the model when message patterns change.


In [97]:
# export model
import joblib

joblib.dump(grid_KNN.best_estimator_, model_dir/"best_knn.pkl")
joblib.dump(grid_SVC.best_estimator_, model_dir/"best_svm.pkl")
joblib.dump(grid_NB.best_estimator_, model_dir/"best_nb.pkl")
all_model_metric.to_csv(output_dir/"model_scores.csv",index=False)